# Camada Bronze - ingestão de dados

Fontes:

- Anatel — Acessos Banda Larga Fixa (SCM), arquivo anual 2026
- IBGE — API de Localidades (municípios e hierarquia geográfica)
- IBGE — API SIDRA (Censo 2022: população e domicílios; PIB municipal)

Princípio da camada: preservar o dado como recebido, sem filtro de conteúdo nem
conversão de tipos. Tudo é lido e gravado como texto. A única alteração é a
normalização dos nomes de coluna para snake_case, exigida pelo formato Delta.

Licença: Anatel e IBGE publicam sob a Lei de Acesso à Informação (Lei 12.527/2011),
livre utilização com citação da fonte.

## Parâmetros

In [0]:
CATALOGO_BRONZE = "bronze"
SCHEMA          = "telecom"
VOLUME_RAW      = f"/Volumes/{CATALOGO_BRONZE}/{SCHEMA}/raw"

In [0]:
ARQUIVO_ANATEL    = "Acessos_Banda_Larga_Fixa_2026.csv"
ARQUIVO_TOTAL     = "Acessos_Banda_Larga_Fixa_Total.csv"
ARQUIVO_DENSIDADE = "Densidade_Banda_Larga_Fixa.csv"

UF_ALVO   = "RS"
UF_CODIGO = "43"   # codigo IBGE do Rio Grande do Sul

Códigos do SIDRA confirmados na etapa de exploração (notebook 00), via API de
metadados `https://servicodados.ibge.gov.br/api/v3/agregados/{id}/metadados`.

**Tabela 4714** — Censo 2022
- `v/93` população residente
- `v/6318` área da unidade territorial
- `v/614` densidade demográfica

**Tabela 4712** — Censo 2022
- `v/381` domicílios particulares permanentes ocupados (denominador da penetração)
- `v/382` moradores nesses domicílios
- `v/5930` média de moradores por domicílio

**Tabela 5938** — PIB dos Municípios
- `v/37` PIB a preços correntes
- `v/543` impostos líquidos de subsídios
- `v/498` VAB total
- `v/513` VAB agropecuária
- `v/517` VAB indústria
- `v/6575` VAB serviços (exceto administração pública)
- `v/525` VAB administração, defesa, educação e saúde públicas

**Ano de referência:** 2021. É o mais recente com o desdobramento setorial completo — em 2022 e 2023 a SIDRA publica apenas o PIB total, e as seis variáveis de VAB e impostos retornam `...` em todos os municípios. Verificado na etapa de exploração.

In [0]:
SIDRA_POPULACAO = {
    "tabela": 4714,
    "variaveis": "93,6318,614",
    "periodo": "2022",
}

SIDRA_DOMICILIOS = {
    "tabela": 4712,
    "variaveis": "381,382,5930",
    "periodo": "2022",
}

SIDRA_PIB = {
    "tabela": 5938,
    "variaveis": "37,543,498,513,517,6575,525",
    # 2021 e o ano mais recente com o desdobramento setorial completo.
    # Em 2022 e 2023 a SIDRA publica apenas o PIB total (v/37); as demais
    # variaveis retornam "..." (dado nao disponivel) em todos os municipios.
    "periodo": "2021",
}

Importações necessárias

In [0]:
import re
import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField

Criação dos catálogos

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO_BRONZE}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO_BRONZE}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOGO_BRONZE}")
spark.sql(f"USE SCHEMA  {SCHEMA}")

print(f"Contexto: {CATALOGO_BRONZE}.{SCHEMA}")

## Funções auxiliares

Converte 'Código IBGE Município' -> 'codigo_ibge_municipio'

In [0]:
def normalizar_nome_coluna(nome: str) -> str:

    acentos = str.maketrans(
        "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
        "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC",
    )
    n = nome.strip().translate(acentos).lower()
    n = re.sub(r"[^a-z0-9]+", "_", n)
    return re.sub(r"_+", "_", n).strip("_")

Leitura do CSV da Anatel preservando tudo como texto.

In [0]:
def ler_csv_bruto(caminho: str):

    return (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("inferSchema", False)
        .csv(caminho)
    )

Padroniza nomes, adiciona metadados de controle e grava como Delta.

In [0]:
def gravar_bronze(df, tabela: str, arquivo_origem: str, fonte: str, comentario: str):

    for antigo in df.columns:
        df = df.withColumnRenamed(antigo, normalizar_nome_coluna(antigo))

    df = (
        df
        .withColumn("_arquivo_origem", F.lit(arquivo_origem))
        .withColumn("_fonte",          F.lit(fonte))
        .withColumn("_data_ingestao",  F.lit(datetime.now(timezone.utc).isoformat()))
    )

    nome_completo = f"{CATALOGO_BRONZE}.{SCHEMA}.{tabela}"
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(nome_completo))

    # ALTERACAO: remove apostrofos para nao quebrar o COMMENT ON TABLE
    spark.sql(f"COMMENT ON TABLE {nome_completo} IS '{comentario.replace(chr(39), '')}'")

    n = spark.table(nome_completo).count()
    print(f"{nome_completo}: {n:,} linhas, {len(df.columns)} colunas")
    return spark.table(nome_completo)

Consulta na API SIDRA para todos os municípios de uma UF.

`n6/in n3 43` significa: nível município, contido na UF de código 43 (RS).

A primeira linha do retorno é cabeçalho descritivo e é descartada.

In [0]:
def baixar_sidra(tabela: int, variaveis: str, periodo: str, uf_codigo: str = UF_CODIGO):

    url = (
        f"https://apisidra.ibge.gov.br/values"
        f"/t/{tabela}/n6/in%20n3%20{uf_codigo}"
        f"/v/{variaveis}/p/{periodo}"
    )
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    dados = r.json()

    cabecalho = dados[0]
    linhas    = dados[1:]
    print(f"Tabela {tabela}: {len(linhas)} registros")

    colunas = {k: normalizar_nome_coluna(v) for k, v in cabecalho.items()}
    registros = [{colunas[k]: str(v) for k, v in linha.items()} for linha in linhas]

    schema = StructType([StructField(c, StringType(), True) for c in registros[0].keys()])
    return spark.createDataFrame(registros, schema=schema), url

## 1. Anatel — Acessos Banda Larga Fixa

In [0]:
df_anatel = ler_csv_bruto(f"{VOLUME_RAW}/{ARQUIVO_ANATEL}")

print("Colunas originais:")
for c in df_anatel.columns:
    print(f"  {c:<30} -> {normalizar_nome_coluna(c)}")

In [0]:
bronze_acessos = gravar_bronze(
    df_anatel,
    tabela="anatel_acessos_raw",
    arquivo_origem=ARQUIVO_ANATEL,
    fonte="Anatel - Dados Abertos - Acessos Banda Larga Fixa (SCM)",
    comentario=(
        "Camada Bronze. Acessos de banda larga fixa declarados pelas prestadoras "
        "de SCM a Anatel. Granularidade: ano x mes x grupo economico x empresa x "
        "CNPJ x UF x municipio x faixa de velocidade x velocidade x tecnologia x "
        "meio de acesso x tipo de pessoa x tipo de produto. Abrangencia nacional, "
        "ano de 2026. Colunas preservadas como texto; nomes normalizados para "
        "snake_case por restricao do formato Delta. "
        "Fonte: https://www.anatel.gov.br/dadosabertos/paineis_de_dados/acessos/"
        "acessos_banda_larga_fixa.zip"
    ),
)

Verificação de meses disponíveis e volume de acessos

In [0]:
display(
    bronze_acessos.groupBy("ano", "mes")
    .agg(F.count("*").alias("linhas"),
         F.sum(F.col("acessos").cast("long")).alias("acessos"))
    .orderBy("ano", F.col("mes").cast("int"))
)

## 2. Anatel — arquivos auxiliares para validação

Validação cruzada para confirmação de referência dos dados.

Total da soma nacional de acessos por mês X densidade calculada pela Anatel por município

In [0]:
gravar_bronze(
    ler_csv_bruto(f"{VOLUME_RAW}/{ARQUIVO_TOTAL}"),
    tabela="anatel_total_raw",
    arquivo_origem=ARQUIVO_TOTAL,
    fonte="Anatel - Dados Abertos",
    comentario=(
        "Camada Bronze. Total nacional de acessos de banda larga fixa por mes, "
        "publicado pela Anatel. Usado como referencia para validar a integridade "
        "da ingestao da tabela anatel_acessos_raw."
    ),
)

gravar_bronze(
    ler_csv_bruto(f"{VOLUME_RAW}/{ARQUIVO_DENSIDADE}"),
    tabela="anatel_densidade_raw",
    arquivo_origem=ARQUIVO_DENSIDADE,
    fonte="Anatel - Dados Abertos",
    comentario=(
        "Camada Bronze. Densidade de acessos de banda larga fixa calculada pela "
        "Anatel, por nivel geografico (Brasil, UF, Municipio). Usada como "
        "referencia para validar a metrica de penetracao calculada na Gold. "
        "Atencao: o separador decimal e virgula."
    ),
)

In [0]:
soma_bronze = (
    spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.anatel_acessos_raw")
    .groupBy("ano", "mes")
    .agg(F.sum(F.col("acessos").cast("long")).alias("soma_bronze"))
)

total_oficial = (
    spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.anatel_total_raw")
    .select("ano", "mes", F.col("acessos").cast("long").alias("total_oficial"))
)

display(
    soma_bronze.join(total_oficial, ["ano", "mes"], "inner")
    .withColumn("diferenca", F.col("soma_bronze") - F.col("total_oficial"))
    .withColumn("integro", F.col("soma_bronze") == F.col("total_oficial"))
    .orderBy("ano", F.col("mes").cast("int"))
)

Validação confirmada.

## 3. IBGE — API de Localidades

Extração dos dados dos municípios por UF, com código IBGE de 7 dígitos

In [0]:
url_localidades = (
    f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_ALVO}/municipios"
)

resp = requests.get(url_localidades, timeout=60)
resp.raise_for_status()
municipios = resp.json()

print(f"Municipios retornados: {len(municipios)}")
print(json.dumps(municipios[0], indent=2, ensure_ascii=False))

Reestruturação dos dados do IBGE

In [0]:
registros = []
for m in municipios:
    micro = m.get("microrregiao") or {}
    meso  = (micro.get("mesorregiao") or {})
    uf    = (meso.get("UF") or {})
    imed  = m.get("regiao-imediata") or {}
    inter = (imed.get("regiao-intermediaria") or {})

    registros.append({
        "codigo_ibge":              str(m["id"]),
        "nome_municipio":           m["nome"],
        "microrregiao_id":          str(micro.get("id", "")),
        "microrregiao_nome":        micro.get("nome", ""),
        "mesorregiao_id":           str(meso.get("id", "")),
        "mesorregiao_nome":         meso.get("nome", ""),
        "regiao_imediata_id":       str(imed.get("id", "")),
        "regiao_imediata_nome":     imed.get("nome", ""),
        "regiao_intermediaria_id":  str(inter.get("id", "")),
        "regiao_intermediaria_nome": inter.get("nome", ""),
        "uf_sigla":                 uf.get("sigla", UF_ALVO),
        "uf_nome":                  uf.get("nome", ""),
    })

schema_loc = StructType([StructField(c, StringType(), True) for c in registros[0].keys()])
df_localidades = spark.createDataFrame(registros, schema=schema_loc)

gravar_bronze(
    df_localidades,
    tabela="ibge_municipios_raw",
    arquivo_origem=url_localidades,
    fonte="IBGE - API de Localidades v1",
    comentario=(
        "Camada Bronze. Municipios da UF com codigo IBGE de 7 digitos e hierarquia "
        "geografica completa (microrregiao, mesorregiao, regiao imediata, regiao "
        "intermediaria). Fonte: API de Localidades do IBGE."
    ),
)

## 4. IBGE — SIDRA

A API SIDRA segue o seguinte padrão:
`https://apisidra.ibge.gov.br/values/t/{tabela}/n6/{municipios}/v/{variavel}/p/{periodo}`
onde `n6` é o nível município. O retorno é um JSON em que a **primeira linha
é o cabeçalho descritivo** e as demais são os dados.

### 4.1 População, área e densidade — Censo 2022 (tabela 4714)

In [0]:
df_pop, url_pop = baixar_sidra(**SIDRA_POPULACAO)
display(df_pop.limit(5))

In [0]:
gravar_bronze(
    df_pop,
    tabela="ibge_populacao_raw",
    arquivo_origem=url_pop,
    fonte="IBGE - API SIDRA - tabela 4714 - Censo Demografico 2022",
    comentario=(
        "Camada Bronze. Populacao residente (v/93), area da unidade territorial "
        "(v/6318) e densidade demografica (v/614) por municipio do RS. "
        "Censo Demografico 2022. Fonte: API SIDRA do IBGE, tabela 4714."
    ),
)

### 4.2 Domicílios — Censo 2022 (tabela 4712)

`v/381` (domicílios particulares permanentes ocupados) é o denominador do mercado
endereçável residencial. Não existe dado público de capacidade instalada de fibra
por município; a contagem de domicílios é o proxy praticável para calcular
penetração.

In [0]:
df_dom, url_dom = baixar_sidra(**SIDRA_DOMICILIOS)
display(df_dom.limit(5))

In [0]:
gravar_bronze(
    df_dom,
    tabela="ibge_domicilios_raw",
    arquivo_origem=url_dom,
    fonte="IBGE - API SIDRA - tabela 4712 - Censo Demografico 2022",
    comentario=(
        "Camada Bronze. Domicilios particulares permanentes ocupados (v/381), "
        "moradores nesses domicilios (v/382) e media de moradores por domicilio "
        "(v/5930), por municipio do RS. Censo Demografico 2022. O campo v/381 e o "
        "denominador usado para calcular penetracao residencial na camada Gold. "
        "Fonte: API SIDRA do IBGE, tabela 4712."
    ),
)

### 4.3 PIB e Valor Adicionado Bruto por setor (tabela 5938)

O maior VAB setorial define objetivamente o setor econômico dominante do município.

Identidade contábil que serve de validação: `PIB = VAB total + impostos`.

Atenção: o PIB Municipal tem defasagem de alguns anos em relação ao snapshot da
Anatel. O ano efetivamente retornado deve ser registrado na documentação como
limitação conhecida.

In [0]:
df_pib, url_pib = baixar_sidra(**SIDRA_PIB)
display(df_pib.limit(5))

In [0]:
gravar_bronze(
    df_pib,
    tabela="ibge_pib_raw",
    arquivo_origem=url_pib,
    fonte="IBGE - API SIDRA - tabela 5938 - PIB dos Municipios",
    comentario=(
        "Camada Bronze. PIB a precos correntes (v/37), impostos liquidos de "
        "subsidios (v/543), VAB total (v/498) e VAB por setor: agropecuaria "
        "(v/513), industria (v/517), servicos exceto administracao publica "
        "(v/6575) e administracao, defesa, educacao e saude publicas (v/525). "
        "Por municipio do RS, ano mais recente disponivel. Valores em mil reais. "
        "Fonte: API SIDRA do IBGE, tabela 5938."
    ),
)

## 5. Inventário e verificação da camada

O RS tem 497 municípios. As quatro fontes devem convergir para esse número.
Divergência indica problema de cobertura, a ser investigado na camada Silver.

In [0]:
tabelas = spark.sql(f"SHOW TABLES IN {CATALOGO_BRONZE}.{SCHEMA}").collect()

print(f"{'TABELA':<25} {'LINHAS':>12}")
print("-" * 38)
for t in sorted(tabelas, key=lambda x: x.tableName):
    nome = f"{CATALOGO_BRONZE}.{SCHEMA}.{t.tableName}"
    print(f"{t.tableName:<25} {spark.table(nome).count():>12,}")

In [0]:
print("Municipios distintos por fonte:")

loc = spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.ibge_municipios_raw")
print(f"  ibge_municipios_raw : {loc.select('codigo_ibge').distinct().count()}")

for tab in ["ibge_populacao_raw", "ibge_domicilios_raw", "ibge_pib_raw"]:
    df = spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.{tab}")
    col_mun = next((c for c in df.columns if "municipio_codigo" in c), None)
    if col_mun:
        print(f"  {tab:<20}: {df.select(col_mun).distinct().count()}")
    else:
        print(f"  {tab:<20}: coluna nao localizada -> {df.columns}")

anatel_rs = (spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.anatel_acessos_raw")
             .filter(F.col("uf") == UF_ALVO))
print(f"  anatel (recorte RS) : {anatel_rs.select('codigo_ibge_municipio').distinct().count()}")

---
**Camada Bronze concluída.**

Próxima etapa: `02_silver_transformacao.py` — recorte RS e mês de referência,
tipagem, tratamento do separador decimal, deduplicação, padronização e validações
de qualidade de dados.